# Automated Web Security Scanner
IBM SkillsBuild Cyber Security 2026 — BharatCares

**Author:** Replace with your full name.

Use only against systems you own or are explicitly authorized to assess. Install dependencies with `!pip install -r requirements.txt` in the project directory before running.

In [ ]:
"""
Automated Web Security Scanner
Educational project for IBM SkillsBuild Cyber Security 2026.

Use only against websites you own or are explicitly authorized to assess.
Checks HTTPS usage, selected security headers, and basic CORS response behavior.
This is a lightweight configuration checker, not a comprehensive penetration tester.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Any
from urllib.parse import urlparse

import requests


SECURITY_HEADERS = {
    "Strict-Transport-Security": (
        "High",
        "Enforces HTTPS in supported browsers. Configure HSTS after confirming HTTPS works.",
    ),
    "Content-Security-Policy": (
        "High",
        "Define a restrictive Content Security Policy appropriate for the application.",
    ),
    "X-Content-Type-Options": (
        "Medium",
        "Set X-Content-Type-Options: nosniff.",
    ),
    "X-Frame-Options": (
        "Medium",
        "Set X-Frame-Options: DENY or SAMEORIGIN, or use CSP frame-ancestors.",
    ),
    "Referrer-Policy": (
        "Low",
        "Set a policy such as strict-origin-when-cross-origin.",
    ),
    "Permissions-Policy": (
        "Low",
        "Explicitly restrict browser features that the application does not need.",
    ),
}


@dataclass
class Finding:
    check: str
    severity: str
    status: str
    evidence: str
    recommendation: str


def validate_url(url: str) -> str:
    """Validate URL shape and reject non-HTTP(S) schemes."""
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.hostname:
        raise ValueError("Enter a complete http:// or https:// URL.")
    if parsed.username or parsed.password:
        raise ValueError("URLs containing embedded credentials are not accepted.")
    return url


def scan_url(url: str, timeout: float = 8.0) -> dict[str, Any]:
    """Perform a small number of ordinary HTTP requests and return a report."""
    url = validate_url(url)
    findings: list[Finding] = []
    checked_at = datetime.now(timezone.utc).isoformat()

    try:
        response = requests.get(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={"User-Agent": "Educational-WebSecurityScanner/1.0"},
        )
    except requests.RequestException as exc:
        return {
            "target": url,
            "final_url": None,
            "status_code": None,
            "checked_at_utc": checked_at,
            "error": str(exc),
            "findings": [
                asdict(Finding(
                    "Connectivity",
                    "Info",
                    "ERROR",
                    str(exc),
                    "Confirm the target is reachable and the URL is correct.",
                ))
            ],
        }

    final_url = response.url
    headers = response.headers

    if urlparse(final_url).scheme == "https":
        findings.append(Finding(
            "HTTPS",
            "Info",
            "PASS",
            f"Final URL uses HTTPS: {final_url}",
            "Keep HTTPS enabled and renew certificates before expiry.",
        ))
    else:
        findings.append(Finding(
            "HTTPS",
            "High",
            "WARN",
            f"Final URL uses HTTP: {final_url}",
            "Serve the site over HTTPS and redirect HTTP requests to HTTPS.",
        ))

    for header, (severity, recommendation) in SECURITY_HEADERS.items():
        value = headers.get(header)
        if value:
            findings.append(Finding(
                f"Security header: {header}",
                "Info",
                "PASS",
                f"{header}: {value}",
                "Review the value to ensure it matches application requirements.",
            ))
        else:
            findings.append(Finding(
                f"Security header: {header}",
                severity,
                "WARN",
                f"{header} was not present in the final response.",
                recommendation,
            ))

    # A simple, non-invasive CORS observation. This is not a full CORS audit.
    origin = "https://scanner-example.invalid"
    try:
        cors_response = requests.get(
            final_url,
            timeout=timeout,
            allow_redirects=True,
            headers={
                "User-Agent": "Educational-WebSecurityScanner/1.0",
                "Origin": origin,
            },
        )
        allow_origin = cors_response.headers.get("Access-Control-Allow-Origin")
        allow_credentials = cors_response.headers.get("Access-Control-Allow-Credentials", "").lower()

        if allow_origin == "*" and allow_credentials == "true":
            findings.append(Finding(
                "CORS configuration",
                "High",
                "REVIEW",
                "Wildcard Access-Control-Allow-Origin and credentials=true were observed.",
                "Review the CORS policy. Do not combine wildcard origins with credentialed access; allow only trusted origins.",
            ))
        elif allow_origin == origin:
            findings.append(Finding(
                "CORS configuration",
                "Medium",
                "REVIEW",
                f"The response reflected the test Origin: {origin}",
                "Confirm that the application uses an explicit trusted-origin allowlist and appropriate Vary: Origin behavior.",
            ))
        elif allow_origin == "*":
            findings.append(Finding(
                "CORS configuration",
                "Low",
                "INFO",
                "Access-Control-Allow-Origin: * was observed.",
                "Confirm that public cross-origin access is intentional and that sensitive responses are not exposed.",
            ))
        else:
            findings.append(Finding(
                "CORS configuration",
                "Info",
                "NO ISSUE OBSERVED",
                f"Access-Control-Allow-Origin: {allow_origin or '(not present)'}",
                "This basic check cannot establish that the complete CORS policy is secure.",
            ))
    except requests.RequestException as exc:
        findings.append(Finding(
            "CORS configuration",
            "Info",
            "ERROR",
            f"CORS observation request failed: {exc}",
            "Review connectivity and rerun if appropriate.",
        ))

    return {
        "target": url,
        "final_url": final_url,
        "status_code": response.status_code,
        "checked_at_utc": checked_at,
        "error": None,
        "findings": [asdict(item) for item in findings],
    }


def write_json(report: dict[str, Any], path: str) -> None:
    with open(path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=2, ensure_ascii=False)


def write_csv(report: dict[str, Any], path: str) -> None:
    fields = ["check", "severity", "status", "evidence", "recommendation"]
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(report.get("findings", []))


def print_report(report: dict[str, Any]) -> None:
    print("\n=== Automated Web Security Scanner ===")
    print(f"Target: {report['target']}")
    print(f"Final URL: {report.get('final_url') or 'Unavailable'}")
    print(f"HTTP status: {report.get('status_code') or 'Unavailable'}")
    print(f"Checked (UTC): {report['checked_at_utc']}")
    if report.get("error"):
        print(f"Connection error: {report['error']}")
    print("\nFindings")
    for item in report.get("findings", []):
        print(f"\n[{item['severity']}] {item['status']} — {item['check']}")
        print(f"  Evidence: {item['evidence']}")
        print(f"  Recommendation: {item['recommendation']}")


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Check basic web security configuration for an authorized target."
    )
    parser.add_argument("url", help="Authorized target URL, e.g. https://example.org")
    parser.add_argument("--timeout", type=float, default=8.0, help="Request timeout in seconds")
    parser.add_argument("--json", default="scan_report.json", help="JSON output path")
    parser.add_argument("--csv", default="scan_report.csv", help="CSV output path")
    args = parser.parse_args()

    if args.timeout <= 0:
        parser.error("--timeout must be greater than zero")

    try:
        report = scan_url(args.url, args.timeout)
    except ValueError as exc:
        print(f"Input error: {exc}", file=sys.stderr)
        return 2

    print_report(report)
    write_json(report, args.json)
    write_csv(report, args.csv)
    print(f"\nSaved: {args.json} and {args.csv}")
    print("Note: results are indicators for review, not proof of exploitability.")
    return 0 if not report.get("error") else 1


if __name__ == "__main__":
    raise SystemExit(main())


## Run
Call `scan_url('https://your-authorized-test-site.example')` after installing `requests`. Review the report before making any security decisions.